# Notebook contrôle qualité des données — Assurance responsabilité civile automobile

Je vérifie les données avant de calculer les KPI. Le but est de repérer les problèmes,
de comprendre leur impact et de noter les décisions prises.

# 1. Importation des données

Les données viennent d'OpenML : `freMTPL2freq` (41214) et `freMTPL2sev` (41215).
J'utilise les fichiers bruts déjà présents dans `data/raw` pour pouvoir relancer le notebook sans téléchargement.

- **freq :** une ligne par contrat. `ClaimNb` est le nombre de sinistres sur sa durée d'exposition.
- **sev :** une ligne par sinistre enregistré, avec son montant. Un contrat peut donc avoir plusieurs lignes.
- **Exposure :** durée d'exposition en années. Ce n'est pas une date.

Les fichiers ARFF commencent par la description des colonnes. Je saute ces lignes pour lire les données avec Pandas.
Les noms des colonnes sont repris dans le même ordre que dans les fichiers.

In [1]:
from pathlib import Path

# Retrouver le dossier du projet depuis la racine ou depuis notebook.
racine = Path.cwd()
if racine.name == 'notebook':
    racine = racine.parent

import pandas as pd

colonnes_freq = [
    'IDpol', 'ClaimNb', 'Exposure', 'Area', 'VehPower', 'VehAge',
    'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Density', 'Region'
]

# 14 lignes de description dans freq, 4 dans sev.
# quotechar retire les apostrophes autour des catégories.
df_freq = pd.read_csv(
    racine / 'data/raw/freMTPL2freq.arff', skiprows=14, names=colonnes_freq,
    quotechar="'", na_values='?', dtype={'IDpol': 'int64'}
)
df_sev = pd.read_csv(
    racine / 'data/raw/freMTPL2sev.arff', skiprows=4, names=['IDpol', 'ClaimAmount'],
    quotechar="'", na_values='?', dtype={'IDpol': 'int64'}
)

# 2. Informations générales et complétude

Je regarde la taille des tables, les valeurs manquantes et la distribution des nombres.
Q1 correspond à 25 %, la médiane à 50 % et Q3 à 75 %. P95 et P99 permettent de regarder les valeurs élevées.

In [2]:
print("="*40 + "freMTPL2freq" + "="*40)
print("Nombre de colonnes :", df_freq.shape[1])
print("Nombre de contrats :", df_freq.shape[0])
print("Valeurs manquantes :")
for col in df_freq.columns:
    print(col, " "*(10 - len(col)), ":", df_freq[col].isna().sum())

print("Distribution des valeurs numériques :")
col_number = [col for col in df_freq.select_dtypes(include=['number']).columns
              if col != 'IDpol']
stats_freq = df_freq[col_number].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
print(stats_freq[['min', 'mean', '25%', '50%', '75%', '95%', '99%', 'max', 'std']].round(2).to_string())
print("Exposition minimale sans arrondi :", df_freq['Exposure'].min())

========================================freMTPL2freq========================================
Nombre de colonnes : 12
Nombre de contrats : 678013
Valeurs manquantes :
IDpol       : 0
ClaimNb     : 0
Exposure    : 0
Area        : 0
VehPower    : 0
VehAge      : 0
DrivAge     : 0
BonusMalus  : 0
VehBrand    : 0
VehGas      : 0
Density     : 0
Region      : 0
Distribution des valeurs numériques :


             min     mean    25%     50%      75%     95%      99%       max      std
ClaimNb      0.0     0.05   0.00    0.00     0.00     1.0      1.0     16.00     0.24
Exposure     0.0     0.53   0.18    0.49     0.99     1.0      1.0      2.01     0.36
VehPower     4.0     6.45   5.00    6.00     7.00    11.0     13.0     15.00     2.05
VehAge       0.0     7.04   2.00    6.00    11.00    17.0     21.0    100.00     5.67
DrivAge     18.0    45.50  34.00   44.00    55.00    72.0     80.0    100.00    14.14
BonusMalus  50.0    59.76  50.00   50.00    64.00    95.0    106.0    230.00    15.64
Density      1.0  1792.42  92.00  393.00  1658.00  7313.0  27000.0  27000.00  3958.65
Exposition minimale sans arrondi : 0.0027322404371584


In [3]:
print("="*40 + "freMTPL2sev" + "="*41)
print("Nombre de colonnes :", df_sev.shape[1])
print("Nombre de lignes de sinistres :", df_sev.shape[0])
print("Nombre de contrats distincts :", df_sev['IDpol'].nunique())
print("Valeurs manquantes :")
for col in df_sev.columns:
    print(col, " "*(12 - len(col)), ":", df_sev[col].isna().sum())

print("Distribution des montants :")
stats_sev = df_sev[['ClaimAmount']].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
print(stats_sev[['min', 'mean', '25%', '50%', '75%', '95%', '99%', 'max', 'std']].round(2).to_string())

========================================freMTPL2sev=========================================
Nombre de colonnes : 2
Nombre de lignes de sinistres : 26639
Nombre de contrats distincts : 24950
Valeurs manquantes :
IDpol         : 0
ClaimAmount   : 0
Distribution des montants :
             min     mean     25%     50%      75%      95%      99%         max       std
ClaimAmount  1.0  2278.54  686.81  1172.0  1228.08  4861.68  16793.7  4075400.56  29297.48


Il n'y a pas de valeur manquante détectée dans les deux tables.
Cela ne veut pas dire que tous les contrats sinistrés ont un montant dans `sev` : je le vérifie dans la partie cohérence.

L'exposition minimale est d'environ **0,00273 année**. L'affichage arrondi à `0.00` ne signifie donc pas qu'elle est nulle.

# 3. Unicité

`IDpol` doit être unique dans `freq`. Dans `sev`, il peut se répéter car un contrat peut avoir plusieurs sinistres.
Je regarde aussi les lignes entièrement identiques.

In [4]:
print("="*40 + "freMTPL2freq" + "="*40)
print("Valeurs uniques :")
print(df_freq.nunique())
print("IDpol répétés après la première occurrence :", df_freq['IDpol'].duplicated().sum())
print("Lignes entièrement répétées :", df_freq.duplicated().sum())

print("="*40 + "freMTPL2sev" + "="*41)
print("Valeurs uniques :")
print(df_sev.nunique())
print("Lignes entièrement répétées :", df_sev.duplicated().sum())

========================================freMTPL2freq========================================
Valeurs uniques :


IDpol         678013
ClaimNb           11
Exposure         181
Area               6
VehPower          12
VehAge            78
DrivAge           83
BonusMalus       115
VehBrand          11
VehGas             2
Density         1607
Region            22
dtype: int64
IDpol répétés après la première occurrence : 0


Lignes entièrement répétées : 0
========================================freMTPL2sev=========================================
Valeurs uniques :
IDpol          24950
ClaimAmount    12369
dtype: int64
Lignes entièrement répétées : 255


Les **678 013 identifiants de `freq` sont uniques**.

Dans `sev`, il y a **255 répétitions exactes après la première occurrence**.
Deux sinistres du même contrat peuvent avoir le même montant. Sans identifiant de sinistre,
je ne peux pas confirmer que ce sont des doublons à supprimer. Je garde donc ces lignes et je note ce point à vérifier.

# 4. Validité

## 4.1. Types des colonnes

Je vérifie les types avant de faire les calculs. Avec la lecture des fichiers locaux,
les catégories sont de type `object` et les identifiants sont des entiers.

In [5]:
df_freq.info()
df_sev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   IDpol       678013 non-null  int64  
 1   ClaimNb     678013 non-null  int64  
 2   Exposure    678013 non-null  float64
 3   Area        678013 non-null  object 
 4   VehPower    678013 non-null  int64  
 5   VehAge      678013 non-null  int64  
 6   DrivAge     678013 non-null  int64  
 7   BonusMalus  678013 non-null  int64  
 8   VehBrand    678013 non-null  object 
 9   VehGas      678013 non-null  object 
 10  Density     678013 non-null  int64  
 11  Region      678013 non-null  object 
dtypes: float64(1), int64(7), object(4)
memory usage: 62.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26639 entries, 0 to 26638
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   IDpol        26639 non-null  int64  
 1  

## 4.2. Valeurs catégorielles

Pour `Area`, `VehBrand` et `Region`, les listes autorisées viennent de la description au début du fichier ARFF.
Pour `VehGas`, je contrôle les deux carburants du jeu de données : `Regular` et `Diesel`.
Les apostrophes ont été retirées lors de l'importation.

In [6]:
print("Contrôle des valeurs catégorielles :")

controls = {
    'Area': ['A', 'B', 'C', 'D', 'E', 'F'],
    'VehGas': ['Regular', 'Diesel'],
    'VehBrand': ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B10', 'B11', 'B12', 'B13', 'B14'],
    'Region': [
        'R11', 'R21', 'R22', 'R23', 'R24', 'R25', 'R26', 'R31', 'R41', 'R42', 'R43',
        'R52', 'R53', 'R54', 'R72', 'R73', 'R74', 'R82', 'R83', 'R91', 'R93', 'R94'
    ]
}

for col in controls:
    n_bad = (~df_freq[col].isin(controls[col])).sum()
    print(f"{col} : {n_bad} valeurs invalides")

Contrôle des valeurs catégorielles :
Area : 0 valeurs invalides


VehGas : 0 valeurs invalides
VehBrand : 0 valeurs invalides
Region : 0 valeurs invalides


## 4.3. Valeurs numériques

Pour calculer la fréquence, l'exposition doit être positive.
Le nombre de sinistres doit être un entier positif ou nul.
Je vérifie aussi les montants non positifs et les âges impossibles.

In [7]:
print("Contrôle des valeurs numériques :")
print("Exposure <= 0 :", (df_freq['Exposure'] <= 0).sum())
print("ClaimNb < 0 :", (df_freq['ClaimNb'] < 0).sum())
# Le reste de la division par 1 est nul pour un entier.
print("ClaimNb non entier :", (df_freq['ClaimNb'] % 1 != 0).sum())
print("ClaimAmount <= 0 :", (df_sev['ClaimAmount'] <= 0).sum())
print("VehAge < 0 :", (df_freq['VehAge'] < 0).sum())
print("DrivAge <= 0 :", (df_freq['DrivAge'] <= 0).sum())
print("Density <= 0 :", (df_freq['Density'] <= 0).sum())

Contrôle des valeurs numériques :
Exposure <= 0 : 0
ClaimNb < 0 : 0
ClaimNb non entier : 0
ClaimAmount <= 0 : 0
VehAge < 0 : 0
DrivAge <= 0 : 0
Density <= 0 : 0


## 4.4. Valeurs à regarder de plus près

Une valeur rare n'est pas forcément une erreur. Je repère les cas particuliers sans les supprimer automatiquement.

In [8]:
print("Contrats avec Exposure > 1 an :", (df_freq['Exposure'] > 1).sum())
print("Exposition maximale :", df_freq['Exposure'].max())
print("Contrats avec VehAge = 100 ans :", (df_freq['VehAge'] == 100).sum())
print("Contrats avec DrivAge = 100 ans :", (df_freq['DrivAge'] == 100).sum())
print("Nombre maximal de sinistres par contrat :", df_freq['ClaimNb'].max())
print("Montant maximal d'un sinistre :", df_sev['ClaimAmount'].max())

Contrats avec Exposure > 1 an : 1224
Exposition maximale : 2.01
Contrats avec VehAge = 100 ans : 25
Contrats avec DrivAge = 100 ans : 3
Nombre maximal de sinistres par contrat : 16
Montant maximal d'un sinistre : 4075400.56


Les contrôles numériques précédents ne détectent pas de valeur invalide selon les règles choisies.
Je garde toutefois ces points à vérifier :

- **1 224 contrats ont une exposition supérieure à un an.** Je dois confirmer la durée couverte. Je ne ramène pas leur exposition à 1.
- **25 véhicules et 3 conducteurs ont un âge de 100 ans.** Je dois vérifier si 100 est un âge réel ou une valeur conventionnelle.
- Le montant maximal est de **4 075 400,56 €**. Il peut peser sur la moyenne, mais son montant élevé ne suffit pas à justifier sa suppression.

Je conserve ces valeurs pour le moment. Je signalerai ces limites lors de l'analyse des KPI.

# 5. Cohérence

## 5.1. Cohérence entre les identifiants des deux tables

Je compare les identifiants pour savoir quels contrats sont présents dans les deux tables et lesquels sont absents.

In [9]:
# Récupérer les ensembles d'IDpol.
ids_freq = set(df_freq['IDpol'].unique())
ids_sev = set(df_sev['IDpol'].unique())

dans_freq_pas_sev = ids_freq - ids_sev
dans_sev_pas_freq = ids_sev - ids_freq
intersection = ids_freq & ids_sev
union = ids_freq | ids_sev

print("="*60)
print("COMPARAISON DES IDpol ENTRE freq ET sev")
print("="*60)
print(f"IDpol distincts dans freq           : {len(ids_freq):>8}")
print(f"IDpol distincts dans sev            : {len(ids_sev):>8}")
print(f"Intersection (dans les deux)        : {len(intersection):>8}")
print(f"Union                              : {len(union):>8}")
print(f"Dans freq seulement                : {len(dans_freq_pas_sev):>8}")
print(f"Dans sev seulement                 : {len(dans_sev_pas_freq):>8}")

COMPARAISON DES IDpol ENTRE freq ET sev
IDpol distincts dans freq           :   678013
IDpol distincts dans sev            :    24950
Intersection (dans les deux)        :    24944
Union                              :   678019
Dans freq seulement                :   653069
Dans sev seulement                 :        6


## 5.2. Contrats de freq absents de sev

Un contrat sans sinistre n'a pas besoin d'avoir une ligne dans `sev`.
Je sépare donc les contrats sans sinistre de ceux qui en déclarent au moins un.

In [10]:
df_freq_sans_sev = df_freq.loc[
    df_freq['IDpol'].isin(dans_freq_pas_sev)
].copy()

sans_sinistre = (df_freq_sans_sev['ClaimNb'] == 0).sum()
avec_sinistre = (df_freq_sans_sev['ClaimNb'] > 0).sum()
nb_contrats_sinistres = (df_freq['ClaimNb'] > 0).sum()
part_sans_montant = avec_sinistre / nb_contrats_sinistres * 100

print("Contrats de freq absents de sev :", len(df_freq_sans_sev))
print("Dont contrats sans sinistre :", sans_sinistre)
print("Dont contrats avec sinistre :", avec_sinistre)
print(f"Part des contrats sinistrés sans ligne dans sev : {part_sans_montant:.2f} %")

Contrats de freq absents de sev : 653069
Dont contrats sans sinistre : 643953
Dont contrats avec sinistre : 9116
Part des contrats sinistrés sans ligne dans sev : 26.76 %


Les **653 069 contrats de `freq` absents de `sev`** se répartissent ainsi :

- **643 953 sans sinistre** : l'absence de ligne dans `sev` est attendue.
- **9 116 avec au moins un sinistre** : les montants ne sont pas disponibles dans le rapprochement.

Ces 9 116 contrats représentent **26,76 % des contrats sinistrés**.
Je les conserve dans `freq` pour calculer la fréquence. Je ne remplace pas leurs coûts manquants par zéro.

## 5.3. Nombre de sinistres par contrat

Pour les contrats présents dans les deux tables, je compare `ClaimNb` au nombre de lignes dans `sev`.
La jointure interne sert uniquement à ce contrôle. Elle ne remplace pas le portefeuille de départ.

In [11]:
# Compter les lignes de sev pour chaque contrat.
nb_sinistres_sev = df_sev.groupby('IDpol').size().reset_index(name='NbSev')

# validate vérifie qu'il n'y a qu'une ligne par IDpol de chaque côté.
comparaison_sinistres = df_freq[['IDpol', 'ClaimNb']].merge(
    nb_sinistres_sev, on='IDpol', how='inner', validate='one_to_one'
)

# Garder les contrats dont les deux nombres sont différents.
ecarts_sinistres = comparaison_sinistres.loc[
    comparaison_sinistres['ClaimNb'] != comparaison_sinistres['NbSev']
].copy()

print("Total ClaimNb dans freq :", df_freq['ClaimNb'].sum())
print("Total des lignes de sev :", len(df_sev))
print("Contrats comparés :", len(comparaison_sinistres))
print("Contrats avec un nombre différent :", len(ecarts_sinistres))
print(ecarts_sinistres.to_string(index=False))

Total ClaimNb dans freq : 36102
Total des lignes de sev : 26639
Contrats comparés : 24944
Contrats avec un nombre différent : 1
  IDpol  ClaimNb  NbSev
4158255        2      1


Sur les **24 944 contrats communs**, un contrat présente un écart :
**4158255** a `ClaimNb = 2`, mais une seule ligne dans `sev`.

Je garde `ClaimNb` tel qu'il est. Le coût observé de ce contrat peut être incomplet.
La présence d'un identifiant dans les deux tables ne garantit donc pas que tous ses sinistres sont renseignés.

# 6. Traitement des contrats orphelins

Je sépare les sinistres dont le contrat n'existe pas dans `freq`.
Ils ne peuvent pas être rattachés à une exposition ou à des caractéristiques de contrat.

In [12]:
ids_orphelins = dans_sev_pas_freq
masque_orphelins = df_sev['IDpol'].isin(ids_orphelins)

# Lignes mises de côté pour investigation.
df_sev_orphelins = df_sev.loc[masque_orphelins].copy()

# Lignes utilisables pour le rapprochement avec freq.
df_sev_rapprochable = df_sev.loc[~masque_orphelins].copy()

montant_total = df_sev['ClaimAmount'].sum()
montant_orphelins = df_sev_orphelins['ClaimAmount'].sum()
montant_rapprochable = df_sev_rapprochable['ClaimAmount'].sum()
part_montant_orphelins = montant_orphelins / montant_total * 100

print("Nombre de contrats orphelins :", df_sev_orphelins['IDpol'].nunique())
print("Nombre de lignes mises de côté :", len(df_sev_orphelins))
print(f"Montant mis de côté : {montant_orphelins:.2f} €")
print(f"Part du montant total de sev : {part_montant_orphelins:.2f} %")
print("Nombre de lignes rapprochables :", len(df_sev_rapprochable))
print(f"Montant rapprochable : {montant_rapprochable:.2f} €")

Nombre de contrats orphelins : 6
Nombre de lignes mises de côté : 195
Montant mis de côté : 788714.18 €
Part du montant total de sev : 1.30 %
Nombre de lignes rapprochables : 26444
Montant rapprochable : 59909216.50 €


Six identifiants de contrat présents dans la table des sinistres sont absents de la table des contrats.
Les **195 lignes** concernées, représentant **788 714,18 €** et **1,30 %** des montants disponibles,
sont exclues des analyses nécessitant un rapprochement avec les caractéristiques ou l'exposition des contrats.

Elles sont conservées dans `df_sev_orphelins` pour investigation. La cause de cette absence n'est pas établie.
Les deux tables importées, `df_freq` et `df_sev`, restent inchangées.

In [13]:
# Vérifier qu'aucune ligne ni aucun montant ne disparaît entre les deux groupes.
print("Répartition des lignes correcte :",
      len(df_sev_orphelins) + len(df_sev_rapprochable) == len(df_sev))

print("Répartition des montants correcte au centime :",
      round(montant_orphelins + montant_rapprochable, 2) == round(montant_total, 2))

print("IDpol absents de freq dans la table rapprochable :",
      (~df_sev_rapprochable['IDpol'].isin(ids_freq)).sum())

Répartition des lignes correcte : True
Répartition des montants correcte au centime : True


IDpol absents de freq dans la table rapprochable : 0


# 7. Synthèse du contrôle qualité

| Contrôle | Résultat | Décision | Impact sur les KPI |
|---|---|---|---|
| Valeurs manquantes dans les tables brutes | 0 dans les deux tables | Pas de remplacement de valeurs manquantes | Cela ne garantit pas la présence des coûts après rapprochement |
| Unicité des contrats dans freq | 678 013 IDpol uniques | Conserver une ligne par contrat | Évite de compter plusieurs fois un contrat dans la base |
| Lignes répétées dans sev | 255 répétitions exactes | Conserver et investiguer, faute d'identifiant de sinistre | Une suppression non justifiée changerait le nombre et le coût des sinistres |
| Catégories autorisées | 0 valeur invalide selon les listes utilisées | Conserver les catégories | Permet de préparer les segments |
| Règles numériques de base | Aucun cas signalé | Conserver les valeurs | Expositions positives, nombres de sinistres entiers et montants positifs |
| Exposition supérieure à un an | 1 224 contrats | Vérifier la durée couverte, sans plafonner à 1 | Le dénominateur de la fréquence dépend de cette durée |
| Âges égaux à 100 ans | 25 véhicules et 3 conducteurs | Conserver et demander la signification de cette valeur | Prudence dans l'analyse par âge |
| Montant maximal | 4 075 400,56 € | Conserver et signaler son poids possible | Peut fortement influencer la moyenne des coûts |
| Contrats de freq sans sinistre et absents de sev | 643 953 contrats | Conserver dans freq | Ils font partie du portefeuille et de l'exposition totale |
| Contrats sinistrés de freq absents de sev | 9 116 contrats, soit 26,76 % des contrats sinistrés | Conserver dans freq ; coûts inconnus | Fréquence calculable à partir de freq, coûts incomplets |
| Écart du nombre de sinistres parmi les contrats communs | IDpol 4158255 : 2 contre 1 | Conserver ClaimNb et signaler l'écart | Le montant observé de ce contrat peut être partiel |
| Contrats de sev absents de freq | 6 contrats, 195 lignes, 788 714,18 € | Séparer dans df_sev_orphelins | 1,30 % des montants de sev hors des analyses par contrat rapproché |
| Répartition après séparation | 195 lignes orphelines + 26 444 rapprochables | Vérifier les lignes et les montants | Pas de perte de données entre les deux groupes |

**Ce que je retiens :** je garde tous les contrats de `freq` pour préparer l'analyse du portefeuille.
Les coûts disponibles ne représentent pas forcément la charge complète des sinistres déclarés.

Pour la suite :

- La fréquence sera calculée avec `SUM(ClaimNb) / SUM(Exposure)`, en signalant le point à vérifier sur les durées.
- Pour relier les coûts aux contrats, j'additionnerai d'abord les montants par `IDpol`, puis je ferai une jointure gauche depuis `freq`.
- Je présenterai les coûts comme des **montants observés**. Je ne remplacerai pas les coûts inconnus des contrats sinistrés par zéro.
- Je garderai les limites identifiées dans la présentation des résultats.